# Scraping Harga Emas dari harga-emas.org
Mengambil harga emas 1 gram (Antam, Pegadaian, Pluang) berdasarkan tanggal.

In [ ]:
# Install dependencies (jalankan sekali saja)
# !pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time

# Mapping bulan ke nama Indonesia (sesuai URL situsnya)
BULAN_ID = {
    1: 'Januari', 2: 'Februari', 3: 'Maret', 4: 'April',
    5: 'Mei', 6: 'Juni', 7: 'Juli', 8: 'Agustus',
    9: 'September', 10: 'Oktober', 11: 'November', 12: 'Desember'
}

In [ ]:
def scrape_harga_emas(tahun, bulan, tanggal):
    """
    Scrape harga emas 1 gram dari harga-emas.org.
    
    Parameters:
        tahun  (int): contoh 2026
        bulan  (int): 1-12
        tanggal (int): 1-31
    
    Returns:
        dict berisi tanggal dan harga Antam, Pegadaian, Pluang (1 gram)
        None jika halaman tidak tersedia / data tidak ditemukan
    """
    nama_bulan = BULAN_ID[bulan]
    url = f"https://harga-emas.org/history-harga/{tahun}/{nama_bulan}/{tanggal:02d}"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            print(f"  [{tahun}/{nama_bulan}/{tanggal:02d}] HTTP {resp.status_code} - skip")
            return None
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        
        # Cari semua baris tabel
        rows = soup.select('table tbody tr')
        
        if not rows:
            print(f"  [{tahun}/{nama_bulan}/{tanggal:02d}] Tabel tidak ditemukan")
            return None
        
        # Baris ke-10 (index 9) = 1 gram
        # Kolom: [0] satuan, [1] Antam, [2] Pegadaian, [3] Pluang
        row_1gram = rows[9]
        cols = row_1gram.find_all('td')
        
        def parse_harga(text):
            """Ubah 'Rp2.796.000' → 2796000 (integer)"""
            return int(text.replace('Rp', '').replace('.', '').replace(',', '').strip())
        
        return {
            'tanggal'   : f"{tahun}-{bulan:02d}-{tanggal:02d}",
            'antam'     : parse_harga(cols[1].text),
            'pegadaian' : parse_harga(cols[2].text),
            'pluang'    : parse_harga(cols[3].text),
        }
    
    except Exception as e:
        print(f"  [{tahun}/{nama_bulan}/{tanggal:02d}] Error: {e}")
        return None

In [ ]:
# ─── Contoh 1: Ambil satu tanggal ───────────────────────────────────────────
data = scrape_harga_emas(2026, 5, 2)
print(data)

In [ ]:
# ─── Contoh 2: Scrape range tanggal (misal 1 bulan penuh) ───────────────────
def scrape_range(start_date: str, end_date: str, delay: float = 1.0):
    """
    Scrape harga emas untuk rentang tanggal.
    
    Parameters:
        start_date (str): format 'YYYY-MM-DD'
        end_date   (str): format 'YYYY-MM-DD'
        delay      (float): jeda antar request dalam detik (default 1s, jangan < 0.5)
    
    Returns:
        DataFrame pandas
    """
    start = datetime.strptime(start_date, '%Y-%m-%d')
    end   = datetime.strptime(end_date,   '%Y-%m-%d')
    
    results = []
    current = start
    
    while current <= end:
        print(f"Fetching {current.strftime('%Y-%m-%d')}...", end=' ')
        row = scrape_harga_emas(current.year, current.month, current.day)
        if row:
            results.append(row)
            print(f"✓ Antam: Rp{row['antam']:,}")
        else:
            print("(tidak ada data)")
        
        current += timedelta(days=1)
        time.sleep(delay)  # jeda agar tidak membebani server
    
    df = pd.DataFrame(results)
    if not df.empty:
        df['tanggal'] = pd.to_datetime(df['tanggal'])
        df = df.sort_values('tanggal').reset_index(drop=True)
    
    return df


# Ambil data April 2026
df = scrape_range('2026-04-01', '2026-04-30')
print(f"\nTotal data: {len(df)} hari")
df.head(10)

In [ ]:
# ─── Simpan ke CSV ──────────────────────────────────────────────────────────
df.to_csv('harga_emas.csv', index=False)
print("Tersimpan ke harga_emas.csv")

# Preview statistik
df[['antam','pegadaian','pluang']].describe().applymap(lambda x: f"Rp{x:,.0f}")